# Face Detection Benchmark Local Workflow

This notebook runs the local RF-DETR labeling workflow for the face detection benchmark. The CLI tools in `src/rfdetr_faces/` are the source of truth; this notebook is a guided wrapper for setup checks, extraction, inference, and COCO export.


## Dataset Policy

The cleaned Roboflow dataset from these target videos is a test-only benchmark. Do not use these images for training, fine-tuning, threshold selection, or augmentation experiments. Train on broader/general face datasets, then evaluate against this target-domain test set.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "rfdetr_faces").exists():
            return candidate
    raise RuntimeError("Could not find repo root from notebook location.")


def repo_path(path: Path) -> Path:
    return path if path.is_absolute() else REPO_ROOT / path


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rfdetr_faces import config as pipeline_config

print(f"Repo root: {REPO_ROOT}")

In [ ]:
import cv2
import torch
from rfdetr import RFDETRLarge

print(f"OpenCV: {cv2.__version__}")
print(f"Torch: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"RFDETRLarge import: {RFDETRLarge}")

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


status_video_dir = repo_path(pipeline_config.DEFAULT_VIDEO_DIR)
status_frames_dir = repo_path(pipeline_config.DEFAULT_FRAMES_DIR)
status_predictions_path = repo_path(pipeline_config.DEFAULT_PREDICTIONS_PATH)
status_coco_export_dir = repo_path(pipeline_config.DEFAULT_ROBOFLOW_EXPORT_DIR)
status_model_path = repo_path(pipeline_config.DEFAULT_MODEL_PATH)

frame_rows = read_jsonl(status_frames_dir / "metadata.jsonl")
prediction_rows = read_jsonl(status_predictions_path)
annotation_path = status_coco_export_dir / "train" / "_annotations.coco.json"

print(f"Model exists: {status_model_path.exists()} - {status_model_path}")
print(f"Videos: {len(list(status_video_dir.glob('*.mp4'))) if status_video_dir.exists() else 0}")
print(f"Extracted frame rows: {len(frame_rows)}")
print(f"Prediction rows: {len(prediction_rows)}")
print(f"Prediction boxes: {sum(len(row.get('detections', [])) for row in prediction_rows)}")
print(f"COCO annotations file exists: {annotation_path.exists()}")

## Run Extraction

Use this step to regenerate sampled frame images from `VIDEO_DIR`. It writes images and `metadata.jsonl` under `EXTRACTED_FRAMES_DIR`. Leave `RUN_EXTRACTION = False` when you only want to inspect existing artifacts.

The variables in the next cell are defaults from the repo config. Replace any path with `repo_path(Path("relative/path"))` or an absolute `Path("/absolute/path")` before running the command cell.


In [ ]:
RUN_EXTRACTION = False
VIDEO_DIR = repo_path(pipeline_config.DEFAULT_VIDEO_DIR)
EXTRACTED_FRAMES_DIR = repo_path(pipeline_config.DEFAULT_FRAMES_DIR)
EXTRACTION_FPS = 3

In [ ]:
if RUN_EXTRACTION:
    subprocess.run(
        [
            "uv",
            "run",
            "rfdetr-faces",
            "extract-frames",
            "--input-dir",
            str(VIDEO_DIR),
            "--output-dir",
            str(EXTRACTED_FRAMES_DIR),
            "--fps",
            str(EXTRACTION_FPS),
            "--overwrite",
        ],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print("Skipping extraction. Set RUN_EXTRACTION = True to run it.")

## Run Inference

Use this step to run the local RF-DETR checkpoint over `INFERENCE_FRAMES_DIR`. It writes detections to `PREDICTIONS_PATH` and optional preview images under `PREVIEW_DIR`. This is the slowest step.

The variables in the next cell are editable. Change `MODEL_PATH`, input/output paths, threshold, or batch size there before setting `RUN_INFERENCE = True`.


In [ ]:
RUN_INFERENCE = False
MODEL_PATH = repo_path(pipeline_config.DEFAULT_MODEL_PATH)
INFERENCE_FRAMES_DIR = EXTRACTED_FRAMES_DIR
PREDICTIONS_PATH = repo_path(pipeline_config.DEFAULT_PREDICTIONS_PATH)
PREVIEW_DIR = repo_path(pipeline_config.DEFAULT_RUNS_DIR) / "previews"
PREDICTION_THRESHOLD = pipeline_config.DEFAULT_CONFIDENCE_THRESHOLD
PREDICTION_BATCH_SIZE = 4

In [ ]:
if RUN_INFERENCE:
    subprocess.run(
        [
            "uv",
            "run",
            "rfdetr-faces",
            "predict-faces",
            "--frames-dir",
            str(INFERENCE_FRAMES_DIR),
            "--output-path",
            str(PREDICTIONS_PATH),
            "--weights",
            str(MODEL_PATH),
            "--threshold",
            str(PREDICTION_THRESHOLD),
            "--batch-size",
            str(PREDICTION_BATCH_SIZE),
            "--preview-dir",
            str(PREVIEW_DIR),
        ],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print("Skipping inference. Set RUN_INFERENCE = True to run it.")

In [ ]:
prediction_rows = read_jsonl(PREDICTIONS_PATH)
empty_images = sum(1 for row in prediction_rows if not row.get("detections"))
box_count = sum(len(row.get('detections', [])) for row in prediction_rows)

print(f"Prediction rows: {len(prediction_rows)}")
print(f"Prediction boxes: {box_count}")
print(f"Images without detections: {empty_images}")

if prediction_rows:
    prediction_rows[0]

## Run COCO Export

Use this step after inference to build the Roboflow-ready COCO folder under `COCO_EXPORT_DIR / "train"`. Keep `INCLUDE_EMPTY_IN_EXPORT = True` for the current test-only benchmark so empty frames are preserved.

The variables in the next cell are editable. Change the frame source, prediction file, or export directory there if you want to create a separate export.


In [ ]:
RUN_EXPORT = False
EXPORT_FRAMES_DIR = INFERENCE_FRAMES_DIR
EXPORT_PREDICTIONS_PATH = PREDICTIONS_PATH
COCO_EXPORT_DIR = repo_path(pipeline_config.DEFAULT_ROBOFLOW_EXPORT_DIR)
INCLUDE_EMPTY_IN_EXPORT = True

In [ ]:
if RUN_EXPORT:
    command = [
        "uv",
        "run",
        "rfdetr-faces",
        "export-coco",
        "--frames-dir",
        str(EXPORT_FRAMES_DIR),
        "--predictions-path",
        str(EXPORT_PREDICTIONS_PATH),
        "--output-dir",
        str(COCO_EXPORT_DIR),
        "--overwrite",
    ]
    if INCLUDE_EMPTY_IN_EXPORT:
        command.append("--include-empty")
    subprocess.run(command, cwd=REPO_ROOT, check=True)
else:
    print("Skipping COCO export. Set RUN_EXPORT = True to run it.")

In [ ]:
annotation_path = COCO_EXPORT_DIR / "train" / "_annotations.coco.json"
if annotation_path.exists():
    coco_data = json.loads(annotation_path.read_text())
    print(f"COCO images: {len(coco_data['images'])}")
    print(f"COCO annotations: {len(coco_data['annotations'])}")
    print(f"COCO categories: {coco_data['categories']}")
else:
    print(f"COCO annotation file not found: {annotation_path}")